# Projeto Fictus | Análise Logística — Bloco 4: Escalabilidade e Riscos Estruturais

---

## Pergunta Central do Bloco
> **Qual modelo sustenta crescimento sem fragilizar a operação — e qual apresenta menor risco estrutural no médio prazo?**

---

## Contexto do Bloco

Escalabilidade e risco são duas faces da mesma moeda. Com a viabilidade e a qualidade mapeadas, este bloco responde qual modelo é mais resiliente sob pressão. Investigamos a fragilidade do sistema diante de picos sazonais e o custo de reversão — caso a estratégia de internalização precise ser desfeita.

Ao identificar o 'Ponto de Ruptura' permite que o comprador saiba até onde o motor logístico aguenta antes de colapsar, transformando riscos operacionais em alavancas de negociação de preço.

**Nota sobre os dados:** o dataset Olist não identifica transportadoras individualmente. A análise de risco de concentração de parceiros logísticos é estruturada a partir de concentração geográfica — que os dados suportam com precisão.

**Este bloco investiga:**
1. O modelo terceirizado atual escala linearmente ou deteriora com o volume?
2. Qual modelo responde melhor a picos sazonais?
3. A concentração geográfica cria risco estrutural — e como o híbrido mitiga?
4. Qual é o custo de reversão se a internalização falhar?
5. Qual é o perfil de sazonalidade das rotas críticas — os picos de volume nas rotas de maior risco coincidem com os picos gerais do negócio, ou são descasados?

---


## Configuração

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import matplotlib.ticker as mticker, matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import warnings
from pathlib import Path
try:
    _base = Path(__file__).resolve().parent
except NameError:
    _base = Path().resolve()
def _find_base(start: Path) -> Path:
    for p in [start, start.parent, start.parent.parent]:
        if (p / "data").exists() or (p / "notebooks").exists():
            return p
    return start
BASE_DIR = _find_base(_base)
DIR_LOG  = BASE_DIR / "data" / "logistics"
DIR_EXPORTS = BASE_DIR / "exports"
DIR_EXPORTS.mkdir(parents=True, exist_ok=True)
warnings.filterwarnings("ignore")
COR_FRETE="#C0392B"; COR_RECEITA="#1B4F72"; COR_MARGEM="#27AE60"
COR_ALERTA="#E74C3C"; COR_NEUTRO="#7F8C8D"; COR_DESTAQUE="#E67E22"
sns.set_theme(style="whitegrid", font_scale=1.0)
plt.rcParams.update({"figure.dpi":150,"savefig.dpi":150,"savefig.bbox":"tight",
    "font.family":"sans-serif","axes.spines.top":False,"axes.spines.right":False})
def fmt_pct(x,pos=None): return f"{x:.1f}%"
def salvar(fig,nome):
    caminho=DIR_EXPORTS/f"{nome}.png"; fig.savefig(caminho); print(f"  -> Salvo: {caminho.name}")
print("Ambiente configurado.")


## Carregamento

In [ ]:
def ler(f,**kw):
    df=pd.read_csv(DIR_LOG/f,low_memory=False,**kw); df.columns=df.columns.str.strip(); return df
log_fato=ler("log_fato.csv"); log_mensal=ler("log_mensal.csv")
log_trim=ler("log_trimestral.csv"); log_rota=ler("log_rota.csv")
for col in ["preco","valor_frete","lead_time_dias","atraso_dias","nota_review",
            "entregue_no_prazo","atrasado"]:
    if col in log_fato.columns: log_fato[col]=pd.to_numeric(log_fato[col],errors="coerce")
# Premissas (auditaveis)
CUSTO_FIXO_MENSAL  = 180_000
CUSTO_VAR_POR_PED  = 12.50
CUSTO_REVERSAO_EST = 300_000
periodos_ord = sorted(log_fato["periodo"].dropna().unique())
print(f"Dados: {len(log_fato):,} pedidos | {periodos_ord[0]} a {periodos_ord[-1]}")


---

## Análise 1 — O modelo terceirizado escala linearmente ou deteriora com o volume?

> *"Um modelo de terceirizacao saudavel deveria ter custo por pedido estavel ou decrescente com o volume. Se a curva e crescente, o modelo esta funcionando como um gargalo disfarçado de servico. Aplico a Teoria das Restricoes para identificar onde esta a restricao real do sistema logistico atual."*

**Framework:** Teoria das Restrições + PDCA  
**Entrega:** Curva de frete por pedido versus volume com identificação do ponto de ruptura

**Como este script responde à pergunta:**
> O script aplica a mesma regressão segmentada do Bloco 3 do Retail — mas agora sobre o frete ao cliente em vez do lead time. Encontra o ponto de volume onde o frete por pedido começa a crescer mais rápido, sinalizando a ruptura da escala do modelo terceirizado.
>
> 1. **Scatter volume × frete por pedido com regressão segmentada:** Dois segmentos de reta — antes e depois do ponto de ruptura. Se o segundo segmento for positivo (frete sobe com volume), a escala do modelo é ruim. Se for plano ou negativo, o modelo escala bem.
> 2. **Evolução do custo por pedido ao longo do tempo:** Série temporal do frete médio por pedido. Tendência crescente confirma deterioração da escala; estável ou decrescente indica modelo ainda saudável.

**Análise do Resultado:**
Esta análise revela o "teto de vidro" da operação. Investigamos se o aumento nas vendas mantém a eficiência ou se gera uma degradação de custos e prazos. Se o modelo deteriora com o volume, ele torna-se uma barreira ao crescimento, sinalizando ao comprador que a escala trará perda de rentabilidade caso a estrutura não seja alterada.


In [ ]:
# Regressao segmentada sobre volume x frete
lm = log_mensal.dropna(subset=["n_pedidos","frete_medio"]).sort_values("n_pedidos").reset_index(drop=True)
x_arr = lm["n_pedidos"].values
y_arr = lm["frete_medio"].values

def reg_segmentada(x,y):
    melhor_r2, melhor_split = -np.inf, None
    for i in range(2, len(x)-2):
        x1,y1=x[:i],y[:i]; x2,y2=x[i:],y[i:]
        if len(x1)<3 or len(x2)<3: continue
        r1=stats.pearsonr(x1,y1)[0]**2; r2=stats.pearsonr(x2,y2)[0]**2
        r2c=(r1*len(x1)+r2*len(x2))/(len(x1)+len(x2))
        if r2c>melhor_r2: melhor_r2,melhor_split=r2c,i
    return melhor_split

split_idx = reg_segmentada(x_arr, y_arr)
vol_ruptura = x_arr[split_idx] if split_idx else None

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Analise 1 - Escala do Modelo Terceirizado: Ponto de Ruptura", fontsize=13, fontweight="bold")

# Scatter segmentado
axes[0].scatter(x_arr, y_arr, color=COR_FRETE, alpha=0.7, s=50, edgecolors="white")
if split_idx and split_idx > 2:
    m1,b1,*_ = stats.linregress(x_arr[:split_idx], y_arr[:split_idx])
    m2,b2,*_ = stats.linregress(x_arr[split_idx:], y_arr[split_idx:])
    xf1 = np.linspace(x_arr[0], x_arr[split_idx-1], 50)
    xf2 = np.linspace(x_arr[split_idx], x_arr[-1], 50)
    axes[0].plot(xf1, m1*xf1+b1, color=COR_MARGEM,  linewidth=2, label=f"Ate {vol_ruptura:,.0f} ped/m")
    axes[0].plot(xf2, m2*xf2+b2, color=COR_FRETE, linewidth=2, label=f"Acima de {vol_ruptura:,.0f}")
    axes[0].axvline(vol_ruptura, color="black", linewidth=1.5, linestyle=":", alpha=0.7)
    dir_escala = "DETERIORA (frete sobe com volume)" if m2 > 0 else "ESCALA BEM (frete cai com volume)"
    axes[0].text(0.02, 0.95, f"Acima do ruptura: {dir_escala}", transform=axes[0].transAxes,
                 fontsize=8, color=COR_ALERTA if m2 > 0 else COR_MARGEM)
axes[0].set_xlabel("Volume mensal de pedidos")
axes[0].set_ylabel("Frete medio ao cliente (R$)")
axes[0].set_title(f"Volume x Frete: Ponto de Ruptura em {vol_ruptura:,.0f} ped/m" if vol_ruptura else "Volume x Frete ao Cliente", fontsize=11)
axes[0].legend(frameon=False, fontsize=8)

# Serie temporal do frete medio
x_t = range(len(log_mensal))
z_fm = np.polyfit(list(x_t), log_mensal["frete_medio"].fillna(method="ffill"), 1)
tend = "deteriorando" if z_fm[0] > 0.01 else "melhorando" if z_fm[0] < -0.01 else "estavel"
axes[1].plot(x_t, log_mensal["frete_medio"], color=COR_FRETE, linewidth=2, marker="o", markersize=3)
axes[1].plot(x_t, np.poly1d(z_fm)(list(x_t)), color="black", linewidth=1, linestyle=":", alpha=0.6)
xtick = list(range(0, len(log_mensal), 3))
axes[1].set_xticks(xtick); axes[1].set_xticklabels([log_mensal["ano_mes"].iloc[i] for i in xtick], rotation=45, ha="right", fontsize=8)
axes[1].set_ylabel("Frete medio ao cliente (R$)")
axes[1].set_title(f"Evolucao do Frete Medio — Tendencia: {tend}", fontsize=11)
plt.tight_layout()
salvar(fig, "13_escala_modelo_ruptura")
plt.show()

print(f"Ponto de ruptura de escala : {vol_ruptura:,.0f} ped/mes" if vol_ruptura else "Ponto de ruptura: nao detectado")
print(f"Volume mensal atual        : {log_mensal['n_pedidos'].mean():,.0f} ped/mes")
print(f"Tendencia temporal frete   : {tend}")
if vol_ruptura:
    print(f"Status                     : {'ACIMA do ponto de ruptura' if log_mensal['n_pedidos'].mean() > vol_ruptura else 'Abaixo do ponto de ruptura'}")


---

## Análise 2 — Qual modelo responde melhor a picos sazonais?

> *"Sazonalidade e frota própria têm uma tensão estrutural: uma operação dimensionada para o pico carrega ociosidade nos meses de vale; uma operação dimensionada para a média colapsa nos picos. A simulação dos dois modelos com os índices de sazonalidade reais revela qual absorve variação de demanda com menor custo e sem deteriorar o SLA."*

**Framework:** PDCA — análise de padrão de demanda + Lean Management  
**Entrega:** Simulação de custo e SLA nos dois modelos durante os meses de pico

**Como este script responde à pergunta:**
> O script usa os índices de sazonalidade calculados por mês para projetar o volume e o frete em cada mês do ano. Compara o custo dos dois modelos mês a mês: o modelo terceirizado tem custo proporcional ao volume; o modelo próprio tem custo fixo dominante mais um componente variável menor, o que o torna mais estável mas potencialmente mais caro nos meses de baixo volume.
>
> 1. **Custo mensal nos dois modelos ao longo do ano:** Barras comparativas de custo para cada mês. Meses de pico são onde a diferença é maior — e onde o modelo terceirizado pode ficar mais caro por pedido.
> 2. **SLA simulado por modelo em meses de pico:** Estima como o SLA de cada modelo se comporta nos meses de maior volume. Operação própria dimensionada para o pico mantém SLA constante; terceirizado pode deteriorar.

**Análise do Resultado:**
Testamos aqui a resiliência do ativo em momentos de estresse (como datas comemorativas). Identificar qual modelo absorve melhor os aumentos repentinos de demanda permite escolher a estrutura com menor risco de colapso operacional e menor perda de reputação em momentos onde o faturamento é vital.


In [ ]:
# Indices de sazonalidade por mes
saz = (
    log_fato.groupby("mes")
    .agg(n_pedidos=("id_pedido","nunique"), frete_medio=("valor_frete","mean"),
         pct_no_prazo=("entregue_no_prazo","mean"))
    .reset_index()
)
saz["pct_no_prazo"] = pd.to_numeric(saz["pct_no_prazo"], errors="coerce") * 100
saz["indice_saz"]   = saz["n_pedidos"] / saz["n_pedidos"].mean()
n_ped_medio = log_mensal["n_pedidos"].mean()

# Custo por mes em cada modelo
saz["vol_estimado"]    = n_ped_medio * saz["indice_saz"]
saz["custo_terc"]      = saz["frete_medio"] * saz["vol_estimado"]
saz["custo_proprio"]   = CUSTO_FIXO_MENSAL + CUSTO_VAR_POR_PED * saz["vol_estimado"]
# SLA simulado: proprio = constante (dimensionado para pico), terc = degrada no pico
sla_base = saz["pct_no_prazo"].mean()
saz["sla_proprio"]     = min(95, sla_base + 8)  # ganho estimado
saz["sla_terc_sim"]    = sla_base - (saz["indice_saz"] - 1).clip(lower=0) * 5  # degrada no pico

MESES_ABREV = {1:"Jan",2:"Fev",3:"Mar",4:"Abr",5:"Mai",6:"Jun",
               7:"Jul",8:"Ago",9:"Set",10:"Out",11:"Nov",12:"Dez"}
saz["mes_abrev"] = saz["mes"].map(MESES_ABREV)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle("Analise 2 - Sazonalidade: Custo e SLA por Modelo ao Longo do Ano", fontsize=13, fontweight="bold")

x_s = range(len(saz))
w = 0.35
axes[0].bar([i-w/2 for i in x_s], saz["custo_terc"]/1000,   width=w, color=COR_FRETE,   alpha=0.8, label="Terceirizado")
axes[0].bar([i+w/2 for i in x_s], saz["custo_proprio"]/1000, width=w, color=COR_RECEITA, alpha=0.8, label="Proprio")
axes[0].set_xticks(x_s); axes[0].set_xticklabels(saz["mes_abrev"], fontsize=9)
axes[0].set_ylabel("Custo estimado mensal (R$ mil)")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f"R$ {v:,.0f}K"))
axes[0].set_title("Custo por Modelo — Variacao Sazonal", fontsize=11)
axes[0].legend(frameon=False, fontsize=8)

axes[1].plot(x_s, saz["sla_proprio"], color=COR_RECEITA, linewidth=2, marker="o", markersize=4, label="Proprio (dimensionado para pico)")
axes[1].plot(x_s, saz["sla_terc_sim"], color=COR_FRETE,   linewidth=2, marker="s", markersize=4, label="Terceirizado (deteriora no pico)")
axes[1].axhline(90, color=COR_NEUTRO, linestyle="--", linewidth=1, alpha=0.7, label="Meta SLA 90%")
axes[1].set_xticks(x_s); axes[1].set_xticklabels(saz["mes_abrev"], fontsize=9)
axes[1].set_ylabel("SLA estimado (%)")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[1].set_title("SLA Simulado por Modelo — Meses de Pico", fontsize=11)
axes[1].legend(frameon=False, fontsize=8)
plt.tight_layout()
salvar(fig, "14_sazonalidade_custo_sla_por_modelo")
plt.show()

mes_pico = saz.loc[saz["indice_saz"].idxmax(), "mes_abrev"]
indice_pico = saz["indice_saz"].max()
print(f"Mes de maior pico         : {mes_pico} (indice {indice_pico:.2f}x a media)")
print(f"Amplitude sazonalidade    : {saz['indice_saz'].max()/saz['indice_saz'].min():.2f}x")
print(f"Custo terceirizado no pico: R$ {saz['custo_terc'].max():,.0f}/mes")
print(f"Custo proprio no pico     : R$ {saz['custo_proprio'].max():,.0f}/mes")
print(f"Custo proprio no vale     : R$ {saz['custo_proprio'].min():,.0f}/mes (fixo domina)")


---

## Análise 3 — A concentração geográfica cria risco estrutural — e como o híbrido mitiga?

> *"A Análise de Vendas identificou que SP, RJ e MG concentram grande parte da receita. Isso é ao mesmo tempo oportunidade e risco: internalizar apenas essas rotas captura a maior parte do benefício, mas cria dependência da operação própria nas regiões mais críticas. O score GUT por estado quantifica onde o risco geográfico é mais concentrado em cada modelo."*

**Framework:** Pareto + Matriz GUT (Gravidade, Urgência, Tendência)  
**Entrega:** Análise de concentração de risco geográfico por modelo com score GUT

**Como este script responde à pergunta:**
> O script calcula o score GUT para cada região: Gravidade (impacto de falha logística na receita), Urgência (quão crítico é o SLA atual nessa região) e Tendência (se o frete ao cliente está crescendo). O heatmap cruza os scores por estado, revelando onde o risco geográfico é mais concentrado em cada modelo.
>
> 1. **Concentração de receita por estado:** Curva de Pareto mostrando quantos estados concentram 80% da receita — e o quanto o modelo híbrido cobre desse núcleo.
> 2. **Heatmap GUT por estado:** Score de risco (G×U×T) para cada estado. Estados com score alto que estão no modelo terceirizado continuam sendo risco; os que entram no modelo híbrido têm risco mitigado.

**Análise do Resultado:**
Aqui mapeamos a "dependência de rotas". Se o faturamento está concentrado em poucas trajetórias, qualquer crise regional ameaça o negócio todo. O modelo híbrido entra como uma estratégia de proteção, diversificando o risco ao garantir controle próprio onde a dependência é maior e a vulnerabilidade é crítica.

In [ ]:
# Pareto de estados
est = (
    log_fato.groupby("estado_cliente")
    .agg(receita=("preco","sum"), n_ped=("id_pedido","nunique"),
         frete_m=("valor_frete","mean"), pct_np=("entregue_no_prazo","mean"))
    .reset_index().sort_values("receita", ascending=False)
)
est["pct_np"]     = pd.to_numeric(est["pct_np"], errors="coerce") * 100
est["pct_receita"]= est["receita"] / est["receita"].sum() * 100
est["pct_acum"]   = est["pct_receita"].cumsum()
n_est_80 = (est["pct_acum"] <= 80).sum() + 1

# Score GUT: G=impacto (pct_receita), U=urgencia (100-pct_np), T=tendencia frete
est["G"] = (est["pct_receita"] / est["pct_receita"].max() * 3).round().clip(1,3).astype(int)
est["U"] = ((100 - est["pct_np"].fillna(80)) / 20).clip(1,3).round().astype(int)
est["T"] = 2  # tendencia moderada (dado fixo — sem serie por estado suficiente)
est["GUT"] = est["G"] * est["U"] * est["T"]
estados_hibrido = ["SP","RJ","MG","PR","RS"]
est["modelo"] = est["estado_cliente"].apply(lambda x: "Hibrido (proprio)" if x in estados_hibrido else "Terceirizado")

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle("Analise 3 - Risco Geografico por Estado: Concentracao e Score GUT", fontsize=13, fontweight="bold")

# Pareto de estados
n_est = len(est)
axes[0].plot(range(n_est), est["pct_acum"].values, color=COR_RECEITA, linewidth=2, marker="o", markersize=3)
axes[0].axhline(80, color=COR_ALERTA, linestyle="--", linewidth=1, label="80% da receita")
axes[0].axvline(n_est_80-1, color=COR_DESTAQUE, linestyle=":", linewidth=1.5)
axes[0].annotate(f"{n_est_80} estados = 80% receita",
                 xy=(n_est_80-1, 80), xytext=(n_est_80+1, 70),
                 fontsize=8, color=COR_DESTAQUE,
                 arrowprops=dict(arrowstyle="->", color=COR_DESTAQUE, lw=1))
cores_bar = [COR_RECEITA if e in estados_hibrido else COR_NEUTRO for e in est["estado_cliente"]]
ax_twin = axes[0].twinx()
ax_twin.bar(range(n_est), est["pct_receita"], color=cores_bar, alpha=0.3)
ax_twin.set_ylabel("% Receita por estado", color=COR_NEUTRO)
axes[0].set_xticks(range(n_est))
axes[0].set_xticklabels(est["estado_cliente"].tolist(), rotation=45, ha="right", fontsize=7)
axes[0].set_ylabel("% Receita Acumulada")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[0].set_title(f"Pareto de Receita por Estado\n(azul = candidatos ao modelo hibrido)", fontsize=11)
axes[0].legend(frameon=False, fontsize=8)

# Heatmap GUT simplificado (top 15 estados)
top15_est = est.head(15)
gut_data  = top15_est[["G","U","T","GUT"]].set_index(top15_est["estado_cliente"])
sns.heatmap(gut_data, ax=axes[1], annot=True, fmt=".0f", cmap="YlOrRd",
            linewidths=0.5, cbar_kws={"label":"Score"})
axes[1].set_title("Score GUT por Estado (Top 15 por receita)\nG=Gravidade | U=Urgencia | T=Tendencia", fontsize=11)
axes[1].tick_params(axis="y", labelsize=8)
plt.tight_layout()
salvar(fig, "15_risco_geografico_gut")
plt.show()

pct_hibrido_cobertura = est[est["estado_cliente"].isin(estados_hibrido)]["pct_receita"].sum()
print(f"Estados para 80% da receita : {n_est_80} de {n_est}")
print(f"Cobertura do modelo hibrido : {pct_hibrido_cobertura:.1f}% da receita")
print(f"Top 3 estados por GUT score:")
for _,r in est.nlargest(3,"GUT").iterrows():
    print(f"  {r['estado_cliente']}: GUT={r['GUT']} | receita:{r['pct_receita']:.1f}% | SLA:{r['pct_np']:.1f}%")


---

## Análise 4 — Qual é o custo de reversão se a internalização falhar?

> *"Toda decisão de alto impacto tem um custo de saída. O capital comprometido em frota e infraestrutura tem custo de oportunidade — e desfazer uma operação própria pode custar mais do que o benefício obtido. A assimetria entre custo de entrada e custo de reversão define a tolerância ao risco de cada cenário e informa qual margem de erro o board está disposto a aceitar."*

**Framework:** Análise de custo de reversão + Matriz de Decisão  
**Entrega:** Estimativa do custo de saída e comparação de assimetria de risco por cenário

**Como este script responde à pergunta:**
> O script compara o perfil de risco de cada cenário em três dimensões: custo de entrada (investimento inicial), custo de reversão (quanto custa desfazer) e janela de decisão (por quanto tempo a decisão pode ser adiada sem custo adicional). A assimetria entre entrada e saída define a tolerância ao risco.
>
> 1. **Comparação entrada × saída por cenário:** Barras mostrando o capital em risco em cada cenário — quanto se compromete na entrada e quanto custa na saída se der errado.
> 2. **Janela de decisão:** Mostra por quanto tempo cada cenário pode ser adiado antes que o custo de postergação supere o custo de agir agora.

**Análise do Resultado:**
Esta é a análise do "Botão de Pânico". Calculamos o passivo financeiro necessário para desfazer a estrutura própria caso a estratégia precise ser revertida. Ter este valor mapeado garante que a decisão de investimento seja tomada com total transparência sobre o risco de capital e a segurança do fluxo de caixa do comprador.

In [ ]:
# Parametros de risco por cenario
CUSTO_IMPLANTACAO = 500_000
cenarios_risco = {
    "Manter terceirizado": {
        "investimento": 0,
        "custo_reversao": 0,
        "custo_postergacao_trim": log_mensal["frete_medio"].mean() * log_mensal["n_pedidos"].mean() * 0.05,
        "descricao": "Sem investimento, mas custo de oportunidade cresce"
    },
    "Modelo hibrido": {
        "investimento": CUSTO_IMPLANTACAO * 0.5,
        "custo_reversao": CUSTO_REVERSAO_EST * 0.4,
        "custo_postergacao_trim": 0,
        "descricao": "Menor risco de capital, cobertura parcial"
    },
    "Internalizar total": {
        "investimento": CUSTO_IMPLANTACAO,
        "custo_reversao": CUSTO_REVERSAO_EST,
        "custo_postergacao_trim": 0,
        "descricao": "Maior capital em risco, maior beneficio potencial"
    }
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Analise 4 - Custo de Reversao e Assimetria de Risco por Cenario", fontsize=13, fontweight="bold")

nomes = list(cenarios_risco.keys())
investimentos = [c["investimento"]/1000 for c in cenarios_risco.values()]
reversoes     = [c["custo_reversao"]/1000 for c in cenarios_risco.values()]

x = range(len(nomes))
w = 0.35
axes[0].bar([i-w/2 for i in x], investimentos, width=w, color=COR_RECEITA, alpha=0.85, label="Capital de entrada")
axes[0].bar([i+w/2 for i in x], reversoes,     width=w, color=COR_FRETE,   alpha=0.85, label="Custo de reversao")
for i, (inv, rev) in enumerate(zip(investimentos, reversoes)):
    if inv > 0: axes[0].text(i-w/2, inv+5, f"R${inv:.0f}K", ha="center", fontsize=8)
    if rev > 0: axes[0].text(i+w/2, rev+5, f"R${rev:.0f}K", ha="center", fontsize=8)
axes[0].set_xticks(x); axes[0].set_xticklabels([n.replace(" ", "\n") for n in nomes], fontsize=9)
axes[0].set_ylabel("Valor (R$ mil)")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f"R$ {v:,.0f}K"))
axes[0].set_title("Capital em Risco: Entrada vs Saida por Cenario", fontsize=11)
axes[0].legend(frameon=False, fontsize=8)

# Razao reversao/investimento (assimetria)
razoes = [r/(i+1) for i,r in zip(investimentos,reversoes)]
cores_r = [COR_MARGEM if r < 0.5 else COR_DESTAQUE if r < 0.8 else COR_FRETE for r in razoes]
bars2 = axes[1].bar(nomes, razoes, color=cores_r, alpha=0.85)
for bar, val in zip(bars2, razoes):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                 f"{val:.2f}x", ha="center", fontsize=9, fontweight="bold")
axes[1].axhline(0.5, color=COR_NEUTRO, linestyle="--", linewidth=1, alpha=0.5, label="Limiar 0.5x")
axes[1].set_xticks(range(len(nomes))); axes[1].set_xticklabels([n.replace(" ", "\n") for n in nomes], fontsize=9)
axes[1].set_ylabel("Razao custo_reversao / investimento")
axes[1].set_title("Assimetria de Risco\n(quanto custa desfazer vs quanto custou fazer)", fontsize=11)
axes[1].legend(frameon=False, fontsize=8)
plt.tight_layout()
salvar(fig, "16_custo_reversao_assimetria")
plt.show()

for nome, c in cenarios_risco.items():
    print(f"{nome}:")
    print(f"  Investimento      : R$ {c['investimento']:,.0f}")
    print(f"  Custo de reversao : R$ {c['custo_reversao']:,.0f}")
    razao = c['custo_reversao']/(c['investimento']+1)
    print(f"  Assimetria        : {razao:.2f}x")


---

## Análise 5 — Qual é o perfil de sazonalidade das rotas críticas — os picos de volume nas rotas de maior receita coincidem?

> *"Sazonalidade distribuída é gerenciável. Sazonalidade concentrada é perigosa. Se as 3 rotas que respondem por 60% da receita atingem o pico no mesmo mês, a operação precisa estar dimensionada para um evento singular — e qualquer falha nesse momento tem impacto desproporcional. Mapear se os picos de volume coincidem entre rotas críticas é definir o dimensionamento correto da operação internalizada."*

**Framework:** Análise de correlação sazonal entre rotas + heatmap de coincidência de picos  
**Entrega:** Heatmap de volume por rota × mês + correlação de picos entre rotas críticas

**Como este script responde à pergunta:**
> O dimensionamento de uma frota própria depende diretamente de saber se os picos de demanda são sincronizados ou distribuídos entre as rotas. Este script mapeia essa dinâmica em dois painéis:
>
> 1. **Heatmap volume por rota × mês:** Cada célula mostra o índice de sazonalidade daquela rota naquele mês — a razão entre o volume daquele mês e a média mensal da rota. Um índice de 1,5 significa que aquele mês tem 50% mais volume que a média. A cor vai do azul claro (abaixo da média) ao azul escuro (pico). Quando várias rotas ficam escuras no mesmo mês, os picos são sincronizados — sinal de que a frota precisa ser dimensionada para o pior caso simultâneo.
> 2. **Correlação de sazonalidade entre rotas críticas:** Calcula a correlação de Pearson entre os perfis sazonais de cada par de rotas críticas. Alta correlação entre rotas de alto volume indica que os picos coincidem e o risco de sobrecarga simultânea é real. Baixa correlação indica que as rotas se complementam — quando uma está no pico, a outra está na baixa — o que permitiria dimensionar a frota de forma mais eficiente.

**Análise do Resultado:**
Esta análise identifica se o risco está "empilhado". Se os problemas nas rotas críticas acontecem ao mesmo tempo que o pico geral de vendas, o risco de ruptura é máximo. Se forem descasados, a empresa tem fôlego operacional para gerenciar os problemas de forma isolada. Este entendimento é crucial para o planejamento de contingência e alocação de frota.

In [ ]:
# ─── Análise 5 — Sazonalidade das Rotas Críticas ─────────────────────────────

# Volume por rota e mês
if "mes" not in log_fato.columns:
    log_fato["mes"] = pd.to_datetime(log_fato["data_compra"], errors="coerce").dt.month

# Top rotas por receita
top_rotas = (
    log_rota.nlargest(10, "receita_total")["rota"].tolist()
    if "rota" in log_rota.columns
    else log_fato.groupby("estado_cliente")["preco"].sum().nlargest(10).index.tolist()
)
col_rota = "rota" if "rota" in log_fato.columns else "estado_cliente"

vol_rota_mes = (
    log_fato[log_fato[col_rota].isin(top_rotas)]
    .groupby([col_rota, "mes"])["id_pedido"]
    .nunique()
    .reset_index(name="n_pedidos")
)

# Índice de sazonalidade por rota
media_rota = vol_rota_mes.groupby(col_rota)["n_pedidos"].transform("mean")
vol_rota_mes["idx_saz"] = vol_rota_mes["n_pedidos"] / media_rota.clip(lower=1)

# Pivot para heatmap
hm_pivot = vol_rota_mes.pivot_table(
    index=col_rota, columns="mes", values="idx_saz", fill_value=1.0
)
# Ordenar por receita
hm_pivot = hm_pivot.reindex(
    [r for r in top_rotas if r in hm_pivot.index]
)
hm_pivot.index = [str(r)[:20] for r in hm_pivot.index]

MESES_NOME = ["Jan","Fev","Mar","Abr","Mai","Jun",
              "Jul","Ago","Set","Out","Nov","Dez"]
hm_pivot.columns = [MESES_NOME[int(c)-1] for c in hm_pivot.columns]

# Correlação entre rotas
corr_rotas = hm_pivot.T.corr()

# Mês de pico coincidente
mes_pico_por_rota = hm_pivot.idxmax(axis=1)
pico_mais_comum   = mes_pico_por_rota.value_counts().idxmax()
n_rotas_pico_comum = mes_pico_por_rota.value_counts().iloc[0]
corr_media = corr_rotas.values[~np.eye(len(corr_rotas), dtype=bool)].mean()

# ─── Plot ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(17, 6))
fig.suptitle("Análise 5 — Sazonalidade das Rotas Críticas: Picos Sincronizados ou Distribuídos?",
             fontsize=13, fontweight="bold")

# Heatmap
im = axes[0].imshow(hm_pivot.values, aspect="auto", cmap="Blues", vmin=0.5, vmax=2.0)
plt.colorbar(im, ax=axes[0], label="Índice de sazonalidade (1.0 = média)", fraction=0.03)
axes[0].set_xticks(range(len(hm_pivot.columns)))
axes[0].set_xticklabels(hm_pivot.columns, fontsize=9)
axes[0].set_yticks(range(len(hm_pivot.index)))
axes[0].set_yticklabels(hm_pivot.index, fontsize=8)
axes[0].set_title("Índice de Sazonalidade por Rota × Mês\n(azul escuro = pico | azul claro = baixa)", fontsize=11)
for r in range(len(hm_pivot.index)):
    for c in range(len(hm_pivot.columns)):
        val = hm_pivot.values[r, c]
        axes[0].text(c, r, f"{val:.1f}", ha="center", va="center",
                     fontsize=7, color="white" if val > 1.4 else "black")

# Heatmap de correlação
im2 = axes[1].imshow(corr_rotas.values, aspect="auto", cmap="RdYlGn", vmin=-1, vmax=1)
plt.colorbar(im2, ax=axes[1], label="Correlação de Pearson", fraction=0.03)
axes[1].set_xticks(range(len(corr_rotas.columns)))
axes[1].set_xticklabels(corr_rotas.columns, rotation=45, ha="right", fontsize=7)
axes[1].set_yticks(range(len(corr_rotas.index)))
axes[1].set_yticklabels(corr_rotas.index, fontsize=7)
axes[1].set_title("Correlação de Sazonalidade entre Rotas\n(verde = picos coincidentes | vermelho = complementares)", fontsize=11)
for r in range(len(corr_rotas)):
    for c in range(len(corr_rotas.columns)):
        val = corr_rotas.values[r, c]
        axes[1].text(c, r, f"{val:.2f}", ha="center", va="center",
                     fontsize=6.5, color="white" if abs(val) > 0.6 else "black")

plt.tight_layout()
salvar(fig, "17_sazonalidade_rotas_criticas")
plt.show()

tipo_pico = "SINCRONIZADO" if corr_media > 0.5 else "DISTRIBUÍDO" if corr_media < 0.2 else "PARCIALMENTE SINCRONIZADO"
print("\n" + "="*55)
print("INSIGHT — SAZONALIDADE DAS ROTAS CRÍTICAS")
print("="*55)
print(f"Rotas analisadas              : {len(hm_pivot)}")
print(f"Mês de pico mais comum        : {pico_mais_comum} ({n_rotas_pico_comum} de {len(hm_pivot)} rotas)")
print(f"Correlação média entre rotas  : {corr_media:.2f}")
print(f"Tipo de pico                  : {tipo_pico}")
if tipo_pico == "SINCRONIZADO":
    print("  → Frota deve ser dimensionada para o pico simultâneo — risco de sobrecarga real")
else:
    print("  → Picos distribuídos permitem dimensionamento mais eficiente da frota própria")


---
## Síntese do Bloco 4 — Escalabilidade e Riscos Estruturais

> **Limitações desta análise:** o ponto de ruptura de escala é estimado por regressão segmentada sobre dados históricos — não representa um limite físico de capacidade da transportadora. O SLA simulado para o modelo próprio em meses de pico assume operação dimensionada para o pico, hipótese que depende de planejamento de capacidade não modelado aqui. O score GUT usa tendência de frete como variável fixa (valor 2) por ausência de série temporal confiável por estado — o que subestima a variação entre regiões. Os custos de reversão são estimativas de benchmark; valores reais dependem de contratos e condições de mercado no momento de eventual desfazimento.


In [ ]:

# Sazonalidade das rotas (análise 5)
_pico_comum_v    = pico_mais_comum
_n_rotas_pico_v  = n_rotas_pico_comum
_corr_media_v    = corr_media
_tipo_pico_v     = tipo_pico

_vol_atual   = log_mensal["n_pedidos"].mean()
_vol_ruptura = vol_ruptura if 'vol_ruptura' in dir() else None
_acima_rup   = _vol_ruptura and _vol_atual > _vol_ruptura
_amplitude   = saz["indice_saz"].max() / saz["indice_saz"].min()
_cob_hibrido = pct_hibrido_cobertura if 'pct_hibrido_cobertura' in dir() else 60.0
_rev_total   = CUSTO_REVERSAO_EST
_rev_hibrido = CUSTO_REVERSAO_EST * 0.4

s_escala = "MODELO ACIMA DO PONTO DE RUPTURA" if _acima_rup else "MODELO ABAIXO DO PONTO DE RUPTURA"
s_saz    = "AMPLITUDE ALTA — frota propria exige planejamento de pico" if _amplitude > 2.0 else "AMPLITUDE MODERADA — gerenciavel"
s_risco  = f"MODELO HIBRIDO COBRE {_cob_hibrido:.1f}% DA RECEITA — mitigacao substancial do risco geografico"

n_alertas = sum([bool(_acima_rup), _amplitude > 2.5])
if n_alertas == 0:
    sinal = "ESCALABILIDADE ADEQUADA — riscos estruturais controlados"
elif n_alertas == 1:
    sinal = "ESCALABILIDADE LIMITADA — um risco estrutural requer atencao"
else:
    sinal = "RISCOS ESTRUTURAIS RELEVANTES — modelo atual proximo do limite"

print("=" * 65)
print("SINTESE - BLOCO 4: ESCALABILIDADE E RISCOS")
print("=" * 65)
print("\n[ ESCALA DO MODELO ]")
print(f"  Volume atual vs ruptura : {s_escala}")
print(f"  Vol. ruptura estimado   : {_vol_ruptura:,.0f} ped/mes" if _vol_ruptura else "  Vol. ruptura           : nao detectado")
print("\n[ SAZONALIDADE ]")
print(f"  Amplitude sazonal       : {_amplitude:.2f}x — {s_saz}")
print(f"  Mes de pico             : {mes_pico if 'mes_pico' in dir() else 'nao calculado'}")
print("\n[ RISCO GEOGRAFICO ]")
print(f"  {s_risco}")
print("\n[ CUSTO DE REVERSAO ]")
print(f"  Internalizar total      : R$ {_rev_total:,.0f} para desfazer")
print(f"  Modelo hibrido          : R$ {_rev_hibrido:,.0f} para desfazer")
print("\n" + "=" * 65)
print("VEREDICTO PARCIAL - BLOCO 4")
print("=" * 65)
print(f"\nSinal geral: {sinal}")
print(f"\n[ SAZONALIDADE DAS ROTAS ]")
print(f"  Mês de pico mais comum  : {_pico_comum_v} ({_n_rotas_pico_v} rotas em pico simultâneo)")
print(f"  Tipo de pico            : {_tipo_pico_v} (corr. média: {_corr_media_v:.2f})")
print("\nProximo passo: Bloco 5 - cenarios de decisao e recomendacao final.")


---
*Próximo notebook: `05_cenarios_recomendacao_log.ipynb` — Fazer, não fazer ou fazer em partes — e qual é o custo de errar em cada direção?*

> Esta análise faz parte do **Projeto Fictus**, conduzido pela Lufi Data Consulting. Os três módulos analíticos — Vendas, Logística e Finanças — compõem a base do Relatório de Recomendação de Aquisição.
